# FITS 文件完整检查

这个 notebook 用来尽可能完整地查看 `data/` 里的 `.fits/.fit/.fts` 文件：

1. 扫描所有 FITS 文件并生成总览表。
2. 展示每个 HDU 的类型、维度、数据类型和完整 header。
3. 自动判断它像是一维光谱、二维图像、表格，还是其它 FITS。
4. 对一维光谱重建波长轴并画谱；对二维图像做快速预览；对表格显示列信息和前几行。

当前这批 `SN2026..._bfosc_*.fits` 很可能是已经抽取/定标好的一维 BFOSC 光谱，不是原始 CCD 图像。运行下面的单元格可以逐个确认。

In [ ]:
%matplotlib inline

import io
import os
import sys
import tempfile
from pathlib import Path
from contextlib import redirect_stdout

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from astropy.io import fits
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## 1. 扫描 `data/` 中的 FITS 文件

In [ ]:
FITS_FILES = sorted(
    p for p in DATA_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".fits", ".fit", ".fts"}
)

print(f"Found {len(FITS_FILES)} FITS files")
for i, path in enumerate(FITS_FILES):
    print(f"[{i:02d}] {path.relative_to(PROJECT_ROOT)}")

## 2. 工具函数

这些函数只读 FITS，不会改动原始文件。

In [ ]:
IMPORTANT_KEYS = [
    "SIMPLE", "BITPIX", "NAXIS", "NAXIS1", "NAXIS2", "NAXIS3",
    "EXTEND", "BZERO", "BSCALE", "BUNIT",
    "OBJECT", "OBJNAME", "TARGET", "DATE", "DATE-OBS", "MJD-OBS",
    "EXPTIME", "EXPOSURE", "AIRMASS", "TELESCOP", "INSTRUME",
    "OBSERVER", "FILTER", "FILTER1", "FILTER2", "GRISM", "GRATING",
    "DISPERSR", "SLIT", "IMAGETYP", "OBSTYPE",
    "RA", "DEC", "OBJCTRA", "OBJCTDEC",
    "CTYPE1", "CTYPE2", "CRVAL1", "CRVAL2", "CRPIX1", "CRPIX2",
    "CDELT1", "CDELT2", "CD1_1", "CD1_2", "CD2_1", "CD2_2",
    "WAT0_001", "WAT1_001", "WAT2_001",
]


def human_size(num_bytes):
    units = ["B", "KB", "MB", "GB"]
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}"
        value /= 1024


def hdu_info_text(hdul):
    buf = io.StringIO()
    with redirect_stdout(buf):
        hdul.info()
    return buf.getvalue()


def header_card_frame(header):
    rows = []
    for card in header.cards:
        rows.append({
            "keyword": card.keyword,
            "value": repr(card.value),
            "comment": card.comment,
        })
    return pd.DataFrame(rows)


def selected_header_values(header):
    return {key: header.get(key, "") for key in IMPORTANT_KEYS if key in header}


def data_shape(data):
    if data is None:
        return ""
    return " x ".join(str(v) for v in np.shape(data))


def data_dtype(data):
    if data is None:
        return ""
    return str(getattr(data, "dtype", ""))


def finite_stats(data):
    if data is None or not hasattr(data, "shape"):
        return {}
    try:
        arr = np.asarray(data)
        if arr.dtype.names:
            return {}
        arr = arr.astype(float, copy=False)
        finite = arr[np.isfinite(arr)]
    except Exception:
        return {}
    if finite.size == 0:
        return {"finite": 0}
    return {
        "finite": int(finite.size),
        "min": float(np.nanmin(finite)),
        "p05": float(np.nanpercentile(finite, 5)),
        "median": float(np.nanmedian(finite)),
        "p95": float(np.nanpercentile(finite, 95)),
        "max": float(np.nanmax(finite)),
    }


def wavelength_axis_from_header(header, n):
    ctype = str(header.get("CTYPE1", "")).upper()
    crval = header.get("CRVAL1")
    crpix = header.get("CRPIX1", 1.0)
    cdelt = header.get("CDELT1", header.get("CD1_1"))
    if crval is None or cdelt is None:
        return None
    pix = np.arange(n, dtype=float) + 1.0
    wave = float(crval) + (pix - float(crpix)) * float(cdelt)
    return wave


def guess_hdu_kind(hdu):
    data = hdu.data
    header = hdu.header
    if data is None:
        return "header-only"
    if getattr(data, "dtype", None) is not None and data.dtype.names:
        return "table"
    shape = np.shape(data)
    naxis = int(header.get("NAXIS", len(shape)))
    ctype1 = str(header.get("CTYPE1", "")).upper()
    bunit = str(header.get("BUNIT", "")).lower()
    if naxis == 1 and ("LINEAR" in ctype1 or "wave" in ctype1.lower() or "/a" in bunit or "angstrom" in bunit):
        return "1D wavelength-calibrated spectrum"
    if naxis == 1:
        return "1D array / possible spectrum"
    if naxis == 2:
        return "2D image / spectral image"
    return f"{naxis}D data"


def summarize_fits_file(path):
    with fits.open(path, memmap=False) as hdul:
        h0 = hdul[0].header
        d0 = hdul[0].data
        stats = finite_stats(d0)
        return {
            "file": str(path.relative_to(PROJECT_ROOT)),
            "size": human_size(path.stat().st_size),
            "hdu_count": len(hdul),
            "kind": guess_hdu_kind(hdul[0]),
            "shape": data_shape(d0),
            "dtype": data_dtype(d0),
            "object": h0.get("OBJECT", h0.get("OBJNAME", "")),
            "date_obs": h0.get("DATE-OBS", ""),
            "exptime": h0.get("EXPTIME", h0.get("EXPOSURE", "")),
            "telescope": h0.get("TELESCOP", ""),
            "instrument": h0.get("INSTRUME", ""),
            "filter/grism": h0.get("FILTER", h0.get("GRISM", h0.get("GRATING", ""))),
            "bunit": h0.get("BUNIT", ""),
            "wave_start": h0.get("CRVAL1", ""),
            "wave_step": h0.get("CDELT1", h0.get("CD1_1", "")),
            "finite": stats.get("finite", ""),
            "median": stats.get("median", ""),
        }

## 3. FITS 总览表

这张表用于快速判断文件大概是什么：一维光谱、二维图像、表格等。

In [ ]:
summary_rows = []
errors = []
for path in FITS_FILES:
    try:
        summary_rows.append(summarize_fits_file(path))
    except Exception as exc:
        errors.append((path, exc))

summary = pd.DataFrame(summary_rows)
display(summary)

if errors:
    print("Files with read errors:")
    for path, exc in errors:
        print(path, repr(exc))

## 4. 选择一个 FITS 详细查看

改 `selected_index` 或直接给 `selected_path` 赋值即可。比如：

```python
selected_index = 0
# 或
selected_path = PROJECT_ROOT / "data/SN2026kid/SN2026kid_bfosc_20260508.fits"
```

In [ ]:
selected_index = 0
selected_path = FITS_FILES[selected_index]

print(f"Selected: [{selected_index}] {selected_path.relative_to(PROJECT_ROOT)}")
print(f"Size:     {human_size(selected_path.stat().st_size)}")

## 5. HDU 结构和完整 header

`HDU` 是 FITS 里的数据单元。一个文件可以只有主 HDU，也可以包含多个图像/表格扩展。下面会打印 `fits.info()`，再把每个 HDU 的 header 完整展开。

In [ ]:
with fits.open(selected_path, memmap=False) as hdul:
    print(hdu_info_text(hdul))

    hdu_rows = []
    for i, hdu in enumerate(hdul):
        hdu_rows.append({
            "hdu": i,
            "name": hdu.name,
            "class": hdu.__class__.__name__,
            "kind": guess_hdu_kind(hdu),
            "shape": data_shape(hdu.data),
            "dtype": data_dtype(hdu.data),
            "header_cards": len(hdu.header.cards),
        })
    display(pd.DataFrame(hdu_rows))

    for i, hdu in enumerate(hdul):
        print("=" * 100)
        print(f"HDU {i}: {hdu.name}  {hdu.__class__.__name__}")
        print("Important header keywords:")
        display(pd.DataFrame([selected_header_values(hdu.header)]).T.rename(columns={0: "value"}))
        print("Full header cards:")
        display(header_card_frame(hdu.header))

## 6. 数据内容检查与预览

- 一维数组：尝试按 FITS header 的 `CRVAL1/CRPIX1/CDELT1` 重建波长轴并画光谱。
- 二维数组：用 zscale/asinh 做快速图像预览。
- 表格：显示列名、格式和前几行。

In [ ]:
with fits.open(selected_path, memmap=False) as hdul:
    for i, hdu in enumerate(hdul):
        data = hdu.data
        print("=" * 100)
        print(f"HDU {i}: {hdu.name}  kind={guess_hdu_kind(hdu)}")
        print(f"shape={data_shape(data)}  dtype={data_dtype(data)}")
        stats = finite_stats(data)
        if stats:
            display(pd.DataFrame([stats]))

        if data is None:
            print("No data array in this HDU.")
            continue

        if getattr(data, "dtype", None) is not None and data.dtype.names:
            print("Table columns:")
            try:
                display(pd.DataFrame({
                    "name": data.names,
                    "format": [data.columns[name].format for name in data.names],
                    "unit": [data.columns[name].unit for name in data.names],
                }))
                display(pd.DataFrame(np.array(data)).head(20))
            except Exception as exc:
                print(f"Could not display table: {exc}")
            continue

        arr = np.asarray(data)
        if arr.ndim == 1:
            y = arr.astype(float)
            x = wavelength_axis_from_header(hdu.header, len(y))
            if x is None:
                x = np.arange(len(y))
                xlabel = "Pixel"
            else:
                xlabel = "Wavelength from FITS WCS (Angstrom if calibrated that way)"

            plt.figure(figsize=(11, 4))
            plt.plot(x, y, lw=0.8)
            plt.xlabel(xlabel)
            plt.ylabel(hdu.header.get("BUNIT", "Flux / counts"))
            plt.title(f"{selected_path.name}  HDU {i}")
            plt.grid(alpha=0.2)
            plt.show()

            print("First 10 points:")
            display(pd.DataFrame({"x": x[:10], "value": y[:10]}))

        elif arr.ndim == 2:
            image = arr.astype(float)
            finite = image[np.isfinite(image)]
            if finite.size:
                try:
                    interval = ZScaleInterval()
                    vmin, vmax = interval.get_limits(image)
                    norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())
                except Exception:
                    vmin, vmax = np.nanpercentile(finite, [1, 99])
                    norm = None
                plt.figure(figsize=(12, 5))
                if norm is not None:
                    plt.imshow(image, origin="lower", cmap="gray", norm=norm, aspect="auto")
                else:
                    plt.imshow(image, origin="lower", cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
                plt.colorbar(label=hdu.header.get("BUNIT", "value"))
                plt.xlabel("X pixel")
                plt.ylabel("Y pixel")
                plt.title(f"{selected_path.name}  HDU {i}")
                plt.show()
            else:
                print("No finite pixels to display.")

        else:
            print("Data has more than 2 dimensions; showing only metadata here.")

## 7. 批量快速画出所有一维光谱

如果文件是一维波长定标光谱，这里会按目标名分组画出来，方便判断哪些是同类数据。

In [ ]:
spectra = []
for path in FITS_FILES:
    try:
        with fits.open(path, memmap=False) as hdul:
            hdu = hdul[0]
            data = hdu.data
            if data is None or np.asarray(data).ndim != 1:
                continue
            y = np.asarray(data, dtype=float)
            x = wavelength_axis_from_header(hdu.header, len(y))
            if x is None:
                x = np.arange(len(y))
            spectra.append({
                "path": path,
                "object": hdu.header.get("OBJECT", path.parent.name),
                "date_obs": hdu.header.get("DATE-OBS", ""),
                "x": x,
                "y": y,
                "bunit": hdu.header.get("BUNIT", ""),
            })
    except Exception as exc:
        print(f"skip {path}: {exc}")

print(f"1D spectra found: {len(spectra)}")

if spectra:
    objects = sorted(set(item["object"] for item in spectra))
    ncols = 2
    nrows = int(np.ceil(len(objects) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, max(4, 3.2 * nrows)), squeeze=False)
    axes_flat = axes.ravel()

    for ax, obj in zip(axes_flat, objects):
        for item in [s for s in spectra if s["object"] == obj]:
            y = item["y"]
            finite = np.isfinite(y)
            scale = np.nanmedian(np.abs(y[finite])) if finite.any() else 1.0
            scale = scale if scale and np.isfinite(scale) else 1.0
            label = Path(item["path"]).stem.replace(f"{obj}_", "")
            ax.plot(item["x"], y / scale, lw=0.8, label=label)
        ax.set_title(obj)
        ax.set_xlabel("Wavelength / pixel")
        ax.set_ylabel("Scaled flux")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)

    for ax in axes_flat[len(objects):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()

## 8. 这个文件大概是什么？

对当前选中文件给一个基于 header 和数据维度的自动解释。这个解释只是辅助判断，最终以 header、数据预览和学长说明为准。

In [ ]:
with fits.open(selected_path, memmap=False) as hdul:
    hdu = hdul[0]
    header = hdu.header
    kind = guess_hdu_kind(hdu)
    print(f"File: {selected_path.relative_to(PROJECT_ROOT)}")
    print(f"Guess: {kind}")
    print()

    if "1D" in kind:
        n = int(header.get("NAXIS1", len(hdu.data)))
        wave = wavelength_axis_from_header(header, n)
        if wave is not None:
            print("Interpretation:")
            print("  这是一个一维数组，而且 header 中有线性波长定标。")
            print("  很可能是已经抽取、波长定标、流量定标后的一维光谱，而不是原始 CCD 图像。")
            print(f"  波长范围约为 {wave[0]:.1f} - {wave[-1]:.1f}，步长约 {np.median(np.diff(wave)):.3f} 每像素。")
            print(f"  Flux unit / BUNIT: {header.get('BUNIT', 'not specified')}")
        else:
            print("Interpretation:")
            print("  这是一个一维数组，但没有足够 WCS 关键字重建波长轴；可能是一维谱或其它序列数据。")
    elif "2D" in kind:
        print("Interpretation:")
        print("  这是二维数据。可能是原始/处理后的 CCD 图像、二维长缝光谱、或练习数据。")
        print("  如果一条光谱在图像中表现为水平/垂直亮带，它还需要抽取成一维谱。")
    elif kind == "table":
        print("Interpretation:")
        print("  这是 FITS 表格扩展，通常用于事件列表、光谱表、catalog 或多列数据。")
    else:
        print("Interpretation:")
        print("  需要结合完整 header 和数据预览进一步判断。")

    print()
    print("Useful header summary:")
    for key in IMPORTANT_KEYS:
        if key in header:
            print(f"  {key:10s} = {header[key]!r}")